# Notebook 03: Quantum EFE vs Classical EFE

**QCCCM** — Quantum Compute for Computational Cognitive Modeling

---

## What you'll learn

1. How **classical Expected Free Energy** (EFE) drives active inference policy selection
2. How **quantum EFE** extends this with density matrices and coherences
3. The `quantumness` parameter: interpolating between classical and quantum agents
4. When quantum interference helps (and hurts) decision-making
5. Side-by-side agent comparison on a simple task

### Prerequisites
- NB 01 (density matrices, entropy)
- NB 02 (quantum walks, interference)
- Basic familiarity with active inference (what EFE is and why we minimize it)

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from qcccm.core.density_matrix import von_neumann_entropy, quantum_relative_entropy
from qcccm.models.bridge import beliefs_to_density_matrix, density_matrix_to_beliefs
from qcccm.models.alf_bridge import (
    alf_quantum_efe,
    beliefs_to_quantum_state,
    evaluate_all_policies,
    preferences_to_density_matrix,
    transition_to_unitary,
)

plt.rcParams['figure.dpi'] = 120
print(f"JAX backend: {jax.default_backend()}")

## 1. Classical Expected Free Energy (Review)

In active inference, an agent selects policies by minimizing the **Expected Free Energy** (EFE):

$$G(\pi) = \underbrace{-\mathbb{E}_{Q(o|\pi)}[\ln P(o)]}_\text{pragmatic: exploit preferences} + \underbrace{\mathbb{E}_{Q(s|\pi)}[H[P(o|s)]]}_\text{epistemic: explore uncertainty}$$

- **Pragmatic term**: Predicted observations should match preferences (exploit)
- **Epistemic term**: Actions should reduce uncertainty about hidden states (explore)

The policy posterior is: $P(\pi) = \sigma(-\gamma \cdot G(\pi))$ (softmax with precision $\gamma$).

In [ ]:
# Minimal classical EFE implementation (no ALF needed)
def classical_efe(A, B, C, beliefs, action):
    """Classical EFE for a single action (one-step lookahead)."""
    eps = 1e-16
    B_a = B[:, :, action]
    q_s_next = B_a @ beliefs
    q_s_next = jnp.clip(q_s_next, eps, None)
    q_s_next = q_s_next / jnp.sum(q_s_next)

    q_o = A @ q_s_next
    q_o = jnp.clip(q_o, eps, None)
    q_o = q_o / jnp.sum(q_o)

    # Pragmatic: E[ln P(o)] where P(o) ∝ exp(C)
    pragmatic = float(jnp.sum(q_o * C))

    # Epistemic: -E[H[P(o|s)]]
    log_A = jnp.log(jnp.clip(A, eps, None))
    entropy_per_state = -jnp.sum(A * log_A, axis=0)
    epistemic = float(-jnp.sum(q_s_next * entropy_per_state))

    return -pragmatic - epistemic

print("Classical EFE: G = -pragmatic - epistemic")
print("Lower G → better policy")

## 2. A Simple Decision Task: The Foraging Problem

Consider an agent in a 3-state environment:
- **State 0**: Safe food source (low reward, certain)
- **State 1**: Risky food source (high reward, uncertain)
- **State 2**: Danger zone (negative reward)

Two actions:
- **Action 0**: Go to safe source (state 0 → state 0, state 1 → state 0)
- **Action 1**: Explore risky source (state 0 → state 1, state 1 → state 1)

The agent must balance exploitation (safe food) vs exploration (risky food).

In [ ]:
# Task setup
n_states = 3
n_obs = 3
n_actions = 2

# A: identity (fully observable)
A = jnp.eye(n_obs, n_states, dtype=jnp.float32)

# B: transition dynamics
B = jnp.zeros((n_states, n_states, n_actions))
# Action 0 (safe): go to state 0
B = B.at[:, :, 0].set(jnp.array([
    [0.9, 0.7, 0.3],  # from any state, likely end up in state 0
    [0.1, 0.2, 0.2],
    [0.0, 0.1, 0.5],
]))
# Action 1 (explore): go to state 1
B = B.at[:, :, 1].set(jnp.array([
    [0.2, 0.1, 0.1],
    [0.7, 0.8, 0.3],  # from most states, likely end up in state 1
    [0.1, 0.1, 0.6],
]))

# C: preferences (log-scale)
C = jnp.array([1.0, 3.0, -5.0])  # prefer state 1 (risky) > state 0 (safe) >> state 2 (danger)

# Initial beliefs (uncertain)
beliefs = jnp.array([0.4, 0.4, 0.2])

print("Task: 3-state foraging problem")
print(f"Preferences (log): {C}")
print(f"Initial beliefs: {beliefs}")

## 3. Classical vs Quantum EFE Comparison

Now we compute EFE for both actions under classical and quantum models.

The quantum model converts beliefs to density matrices, adds coherences
(controlled by the `quantumness` parameter), and evaluates EFE using
von Neumann entropy and quantum relative entropy.

In [ ]:
# Compare classical and quantum EFE across quantumness values
quantumness_values = np.linspace(0, 0.8, 20)

G_classical = np.zeros((2, len(quantumness_values)))
G_quantum = np.zeros((2, len(quantumness_values)))

for i, q in enumerate(quantumness_values):
    for action in range(2):
        G_classical[action, i] = classical_efe(A, B, C, beliefs, action)
        G_quantum[action, i] = float(alf_quantum_efe(A, B, C, beliefs, action, quantumness=q))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# EFE values
for action, (color, label) in enumerate(zip(['blue', 'red'], ['Safe (a=0)', 'Explore (a=1)'])):
    axes[0].plot(quantumness_values, G_quantum[action], f'{color[0]}-', linewidth=2, label=f'Quantum: {label}')
    axes[0].axhline(y=G_classical[action, 0], color=color, linestyle='--', alpha=0.5, label=f'Classical: {label}')

axes[0].set_xlabel('Quantumness (q)')
axes[0].set_ylabel('Expected Free Energy G')
axes[0].set_title('EFE vs Quantumness')
axes[0].legend(fontsize=9)

# Policy probabilities (softmax)
gamma = 4.0
for i, q in enumerate(quantumness_values):
    G_q = np.array([G_quantum[0, i], G_quantum[1, i]])
    probs = np.exp(-gamma * G_q)
    probs = probs / probs.sum()
    G_quantum[0, i] = probs[0]  # reuse array for probs
    G_quantum[1, i] = probs[1]

# Also classical policy probs
G_c = np.array([G_classical[0, 0], G_classical[1, 0]])
c_probs = np.exp(-gamma * G_c)
c_probs = c_probs / c_probs.sum()

axes[1].plot(quantumness_values, G_quantum[0], 'b-', linewidth=2, label='P(safe)')
axes[1].plot(quantumness_values, G_quantum[1], 'r-', linewidth=2, label='P(explore)')
axes[1].axhline(y=c_probs[0], color='blue', linestyle='--', alpha=0.5, label=f'Classical P(safe)={c_probs[0]:.2f}')
axes[1].axhline(y=c_probs[1], color='red', linestyle='--', alpha=0.5, label=f'Classical P(explore)={c_probs[1]:.2f}')
axes[1].set_xlabel('Quantumness (q)')
axes[1].set_ylabel('Policy probability')
axes[1].set_title('Policy Selection vs Quantumness (γ=4)')
axes[1].legend(fontsize=9)

plt.suptitle('Quantum Coherence Modulates Policy Selection', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. What's Happening: Density Matrix View

Let's peek inside the quantum computation to see how coherences change the predicted state and thus the EFE.

In [ ]:
# Visualise density matrices at different quantumness levels
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, q in enumerate([0.0, 0.3, 0.7]):
    # Belief state
    rho = beliefs_to_quantum_state(beliefs, quantumness=q)
    S = von_neumann_entropy(rho)

    im = axes[0, col].imshow(np.abs(np.asarray(rho)), vmin=0, vmax=0.5, cmap='Blues')
    axes[0, col].set_title(f'Belief state (q={q})\nS={float(S):.3f}')
    axes[0, col].set_xticks(range(3))
    axes[0, col].set_yticks(range(3))
    for i in range(3):
        for j in range(3):
            axes[0, col].text(j, i, f'{abs(float(rho[i,j])):.2f}',
                            ha='center', va='center', fontsize=11)

    # Predicted state after action 1 (explore)
    B_a = B[:, :, 1]
    U = transition_to_unitary(B_a)
    rho_pred = U @ rho @ jnp.conj(U).T
    S_pred = von_neumann_entropy(rho_pred)

    im2 = axes[1, col].imshow(np.abs(np.asarray(rho_pred)), vmin=0, vmax=0.5, cmap='Reds')
    axes[1, col].set_title(f'Predicted (explore, q={q})\nS={float(S_pred):.3f}')
    axes[1, col].set_xticks(range(3))
    axes[1, col].set_yticks(range(3))
    for i in range(3):
        for j in range(3):
            axes[1, col].text(j, i, f'{abs(float(rho_pred[i,j])):.2f}',
                            ha='center', va='center', fontsize=11)

fig.suptitle('Density Matrices: Classical (q=0) → Quantum (q=0.7)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Multi-Step Agent Simulation

Let's simulate a classical and quantum agent over multiple trials on a simple environment and compare their behavior.

In [ ]:
# Simple environment simulation
def run_agent_simulation(A, B, C, beliefs_init, n_trials=50, quantumness=0.0, gamma=4.0):
    """Run an agent on the foraging task for n_trials."""
    beliefs = beliefs_init.copy()
    history = {'actions': [], 'beliefs': [], 'G': [], 'rewards': []}

    policies = np.array([[[0]], [[1]]])  # single-step policies

    for t in range(n_trials):
        # Evaluate policies
        G = np.zeros(2)
        for a in range(2):
            G[a] = float(alf_quantum_efe(A, B, C, beliefs, a, quantumness=quantumness))

        # Softmax action selection (deterministic for reproducibility)
        probs = np.exp(-gamma * G)
        probs = probs / probs.sum()
        action = int(np.argmax(probs))  # greedy for clarity

        # Transition
        B_a = np.asarray(B[:, :, action])
        next_beliefs = B_a @ np.asarray(beliefs)
        next_beliefs = np.clip(next_beliefs, 1e-16, None)
        next_beliefs = next_beliefs / next_beliefs.sum()

        # Reward (based on where we end up)
        state = np.random.choice(3, p=next_beliefs)
        reward = [1.0, 3.0, -5.0][state]

        history['actions'].append(action)
        history['beliefs'].append(np.asarray(beliefs).copy())
        history['G'].append(G.copy())
        history['rewards'].append(reward)

        beliefs = jnp.array(next_beliefs)

    return history

np.random.seed(42)
hist_classical = run_agent_simulation(A, B, C, beliefs, quantumness=0.0)
np.random.seed(42)
hist_quantum = run_agent_simulation(A, B, C, beliefs, quantumness=0.4)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Actions over time
axes[0, 0].step(range(50), hist_classical['actions'], 'b-', alpha=0.8, label='Classical', where='mid')
axes[0, 0].step(range(50), hist_quantum['actions'], 'r-', alpha=0.8, label='Quantum (q=0.4)', where='mid')
axes[0, 0].set_ylabel('Action (0=safe, 1=explore)')
axes[0, 0].set_title('Action Selection Over Time')
axes[0, 0].legend()
axes[0, 0].set_yticks([0, 1])

# Cumulative reward
cum_c = np.cumsum(hist_classical['rewards'])
cum_q = np.cumsum(hist_quantum['rewards'])
axes[0, 1].plot(cum_c, 'b-', linewidth=2, label='Classical')
axes[0, 1].plot(cum_q, 'r-', linewidth=2, label='Quantum (q=0.4)')
axes[0, 1].set_ylabel('Cumulative reward')
axes[0, 1].set_title('Cumulative Reward')
axes[0, 1].legend()

# EFE difference (explore - safe) over time
efe_diff_c = [g[1] - g[0] for g in hist_classical['G']]
efe_diff_q = [g[1] - g[0] for g in hist_quantum['G']]
axes[1, 0].plot(efe_diff_c, 'b-', linewidth=2, label='Classical')
axes[1, 0].plot(efe_diff_q, 'r-', linewidth=2, label='Quantum (q=0.4)')
axes[1, 0].axhline(y=0, color='k', linewidth=0.5)
axes[1, 0].set_xlabel('Trial')
axes[1, 0].set_ylabel('G(explore) - G(safe)')
axes[1, 0].set_title('EFE Difference (< 0 favors explore)')
axes[1, 0].legend()

# Belief evolution (state 1 probability)
b1_c = [b[1] for b in hist_classical['beliefs']]
b1_q = [b[1] for b in hist_quantum['beliefs']]
axes[1, 1].plot(b1_c, 'b-', linewidth=2, label='Classical')
axes[1, 1].plot(b1_q, 'r-', linewidth=2, label='Quantum (q=0.4)')
axes[1, 1].set_xlabel('Trial')
axes[1, 1].set_ylabel('P(state 1)')
axes[1, 1].set_title('Belief in Risky State')
axes[1, 1].legend()

plt.suptitle('Classical vs Quantum Agent on Foraging Task', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Quantumness Sweep: Optimal Coherence

Is there an optimal level of quantumness? Too little → classical (may get stuck). Too much → noisy interference. Let's sweep.

In [ ]:
# Sweep quantumness and measure total reward
q_values = np.linspace(0, 0.8, 15)
mean_rewards = []

for q in q_values:
    rewards_across_seeds = []
    for seed in range(10):
        np.random.seed(seed)
        hist = run_agent_simulation(A, B, C, beliefs, quantumness=q, n_trials=50)
        rewards_across_seeds.append(np.sum(hist['rewards']))
    mean_rewards.append(np.mean(rewards_across_seeds))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(q_values, mean_rewards, 'ko-', linewidth=2, markersize=6)
best_q = q_values[np.argmax(mean_rewards)]
ax.axvline(x=best_q, color='red', linestyle='--', alpha=0.7, label=f'Best q = {best_q:.2f}')
ax.set_xlabel('Quantumness (q)')
ax.set_ylabel('Mean total reward (50 trials)')
ax.set_title('Optimal Quantumness for the Foraging Task')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Best quantumness: q = {best_q:.2f}")
print(f"Classical reward (q=0): {mean_rewards[0]:.1f}")
print(f"Best quantum reward: {max(mean_rewards):.1f}")

## Key Takeaways

1. **Quantum EFE generalises classical EFE**: At q = 0, they are equivalent. The quantum version adds one new ingredient — off-diagonal coherences — that enables interference between decision paths.

2. **The quantumness parameter** controls how "quantum" the agent's policy evaluation is. It can be fit to behavioral data using `qcccm.fitting`.

3. **Optimal quantumness is task-dependent**: Some tasks benefit from quantum interference (faster exploration, escape from local optima). Others don't. This mirrors the empirical finding that human decision-making shows quantum-like effects in specific contexts (conjunction fallacy, order effects, sure-thing violations) but not universally.

4. **Connection to neuroscience**: The quantumness parameter can be interpreted as the strength of neural coherence that survives decoherence. Fast decisions (short decoherence time) are more classical; slow, deliberative decisions may show more quantum-like interference.

---

## Exercises

### Basic

**Exercise 3.1:** Modify the preferences C to make state 0 (safe) strongly preferred. How does quantumness affect policy selection when the safe option is obviously best?

**Exercise 3.2:** Create a task where the transition matrices are identity (no state changes). In this case, quantum and classical EFE should be more similar. Verify this.

### Stretch

**Exercise 3.3 (Sure-Thing Principle):** Design a 2-state, 2-action task where the classical agent always chooses the same action regardless of which state it's in (the "sure thing"). Show that the quantum agent can violate this — choosing different actions depending on its quantum state, even when the classical marginals are the same. This requires q > 0 and appropriate transition matrices.

**Exercise 3.4 (ALF integration):** If you have ALF installed, create a `GenerativeModel` and run both `AnalyticAgent` and `QuantumEFEAgent` from `qcccm.models.alf_bridge` on the same task. Compare their learning curves over 100 trials.

## Further Reading

- **Busemeyer & Bruza (2012)** — *Quantum Models of Cognition and Decision*
- **Friston et al. (2022)** — "Active inference and the free energy principle"
- **Pothos & Busemeyer (2022)** — "Quantum cognition" in *Annual Review of Psychology*
- **Khrennikov (2010)** — *Ubiquitous Quantum Structure*